# Chapter 5 Lab — Dedup & Quality Filtering (MinHash near-duplicates)

**Estimated time:** 20 minutes · **Read-and-run.** Pure-Python MinHash; no installs.

In [ ]:
import hashlib, random
random.seed(0)
corpus = [
 'The quick brown fox jumps over the lazy dog',
 'The quick brown fox jumped over a lazy dog',   # near-dup of #0
 'Completely unrelated text about GPU scheduling',
 'The quick brown fox jumps over the lazy dog.',  # near-dup of #0
 'aaa aaa aaa aaa aaa',                           # low quality (repetitive)
]
def shingles(text, k=3):
    toks=text.lower().split()
    return set(tuple(toks[i:i+k]) for i in range(max(1,len(toks)-k+1)))

In [ ]:
NUM_HASH=64
seeds=[random.randrange(1<<30) for _ in range(NUM_HASH)]
def h(x, seed):
    return int(hashlib.md5((str(seed)+str(x)).encode()).hexdigest(),16)
def minhash(sig_set):
    return tuple(min(h(s,seed) for s in sig_set) for seed in seeds) if sig_set else tuple([0]*NUM_HASH)
def est_jaccard(a,b):
    return sum(1 for x,y in zip(a,b) if x==y)/NUM_HASH

sigs=[minhash(shingles(t)) for t in corpus]
print('pairwise estimated Jaccard (near-dup if >= 0.6):')
for i in range(len(corpus)):
    for j in range(i+1,len(corpus)):
        s=est_jaccard(sigs[i],sigs[j])
        flag=' <= NEAR-DUP' if s>=0.6 else ''
        if s>0: print(f'  doc{i} vs doc{j}: {s:.2f}{flag}')

In [ ]:
# Simple quality filters.
def quality_ok(text):
    toks=text.lower().split()
    if len(toks) < 4: return False, 'too short'
    uniq=len(set(toks))/len(toks)
    if uniq < 0.4: return False, 'repetitive'
    return True, 'ok'
for i,t in enumerate(corpus):
    ok,why=quality_ok(t); print(f'doc{i}: {"KEEP" if ok else "DROP"} ({why})')

## Key takeaway
Near-duplicate removal (MinHash/Jaccard) and quality filtering are the foundation every
customization technique depends on. The exam tests recognizing data-quality failure modes —
duplication inflates training cost and memorization; low-diversity text degrades the model.